# Harris corner manual calculation

---

### Dependencies

In [1]:
%pip install numpy Pillow

import numpy as np
from PIL import Image
from src.utils import round_matrix
import src.harris as harris

Note: you may need to restart the kernel to use updated packages.


---

## Given

#### Image $I$

$ I = 
\begin{bmatrix}
49 & 223 & 46 & 174 & 128 \\
84 & 165 & 223 & 233 & 97 \\
168 & 87 & 187 & 42 & 62 \\
115 & 128 & 100 & 202 & 3 \\
71 & 89 & 76 & 50 & 244
\end{bmatrix}
$

In [2]:
I = np.array([
    [49 , 223 , 46 , 174 , 128],
    [84 , 165 , 223 , 233 , 97],
    [168 , 87 , 187 , 42 , 62],
    [115 , 128 , 100 , 202 , 3],
    [71 , 89 , 76 , 50 , 244]
])

image_width, image_height = len(I[0]), len(I)

I

array([[ 49, 223,  46, 174, 128],
       [ 84, 165, 223, 233,  97],
       [168,  87, 187,  42,  62],
       [115, 128, 100, 202,   3],
       [ 71,  89,  76,  50, 244]])

#### BorderBoundaryPadding
Then the extending matrix using repeated at the border (BorderBoundaryPadding). This means that at the four edges of the image, as well as at the four corner locations, the pixel intensities are extended (copied) to the outside of the image.

$ I_{extended} = 
\begin{bmatrix}
\underline{49} & \underline{49} & \underline{223} & \underline{46} & \underline{174} & \underline{128} & \underline{128} \\
\underline{49} & 49 & 223 & 46 & 174 & 128 & \underline{128} \\
\underline{84} & 84 & 165 & 223 & 233 & 97 & \underline{97} \\
\underline{168} & 168 & 87 & 187 & 42 & 62 & \underline{62} \\
\underline{115} & 115 & 128 & 100 & 202 & 3 & \underline{3} \\
\underline{71} & \textbf{71} & 89 & 76 & 50 & 244 & \underline{244} \\
\underline{71} & \underline{71} & \underline{89} & \underline{76} & \underline{50} & \underline{244} & \underline{244} 
\end{bmatrix}
$

In [3]:
I_extended = np.pad(I, pad_width=1, mode="edge")
I_extended

array([[ 49,  49, 223,  46, 174, 128, 128],
       [ 49,  49, 223,  46, 174, 128, 128],
       [ 84,  84, 165, 223, 233,  97,  97],
       [168, 168,  87, 187,  42,  62,  62],
       [115, 115, 128, 100, 202,   3,   3],
       [ 71,  71,  89,  76,  50, 244, 244],
       [ 71,  71,  89,  76,  50, 244, 244]])

## Step 1: Gaussian filter
The filter kernel is:
$1/16
\begin{bmatrix}
1 & 2 & 1 \\
2 & 4 & 2 \\
1 & 2 & 1
\end{bmatrix}
$

#### Calculation

The value at bottom left, aka **71**: 

Look at the bottom corner of $I_{extended}$

$ I_{extended} = 
\begin{bmatrix}
\underline{...} & ... & ... & ... \\
\underline{115} & 115 & 128 & ... \\
\underline{71} & \textbf{71} & 89 & ... \\
\underline{71} & \underline{71} & \underline{89} & ...
\end{bmatrix}
$

$
(71*(1+4+2+2) + 115*(1+2) + 128 + 89*(1+ 2)/16=1379/16=86.1875
$

In [4]:
g_I = harris.gaussian_filtering(I, image_width, image_height)

print("Bottom left corner of I:", g_I[4][0])

print("\nFiltered I:")
print(g_I)

r_g_I = round_matrix(g_I)
print("\nRounded Filtered I:")
print(r_g_I)

Bottom left corner of I: 86.1875

Filtered I:
[[ 95.4375 141.25   144.4375 147.     137.375 ]
 [112.1875 146.5    167.5    151.6875 114.625 ]
 [129.5    135.375  148.75   122.4375  74.4375]
 [114.9375 112.25   115.875  110.4375  89.5   ]
 [ 86.1875  90.375   87.6875 110.4375 159.8125]]

Rounded Filtered I:
[[ 95 141 144 147 137]
 [112 147 168 152 115]
 [130 135 149 122  74]
 [115 112 116 110  90]
 [ 86  90  88 110 160]]


## Step 2: Compute the image gradients

Compute the image gradients in the x direction (Ix) using the following Sobel kernels:

Vertical:
$s_x = 1/8
\begin{bmatrix}
1 & 0 & -1 \\
2 & 0 & -2 \\
1 & 0 & -1
\end{bmatrix}
$

Horizontal:
$s_y = 1/8
\begin{bmatrix}
1 & 2 & 1 \\
0 & 0 & 0 \\
-1 & -2 & -1
\end{bmatrix}
$

#### Calculalation:
the vertical value at bottom left, aka **71**: 

Look at the bottom corner of $I_{extended}$

$ I_{extended} = 
\begin{bmatrix}
\underline{...} & ... & ... & ... \\
\underline{115} & 115 & 112 & ... \\
\underline{86} & \textbf{86} & 90 & ... \\
\underline{86} & \underline{86} & \underline{90} & ...
\end{bmatrix}
$

$
(86*(1+2) + 115 + 112*(-1) + 90*(-1 - 2))/8=-9/8=-1.125
$

In [5]:
Ix = harris.sobel_derivative(r_g_I, image_width, image_height, direction="x")
Iy = harris.sobel_derivative(r_g_I, image_width, image_height, direction="y")

print("Bottom left corner of Ix:", Ix[4][0])
# print("Bottom left corner of Iy:", Iy[4][0])

print("\nFiltered Ix:")
print(Ix)

r_Ix = round_matrix(Ix)
print("\nRounded Filtered Ix:")
print(r_Ix)

print("\nFiltered Iy:")
print(Iy)

r_Iy = round_matrix(Iy)
print("\nRounded Filtered Iy:")
print(r_Iy)

Bottom left corner of Ix: -1.125

Filtered Ix:
[[-21.625 -25.375  -2.875   9.25    8.375]
 [-15.125 -22.5    -0.375  23.5    16.5  ]
 [ -5.25  -11.875   2.875  28.625  19.125]
 [ -0.375  -2.875  -0.375   6.875   4.75 ]
 [ -1.125  -0.875  -7.25  -23.75  -16.25 ]]

Rounded Filtered Ix:
[[-22 -25  -3   9   8]
 [-15 -23   0  24  17]
 [ -5 -12   3  29  19]
 [  0  -3   0   7   5]
 [ -1  -1  -7 -24 -16]]

Filtered Iy:
[[ -7.125  -6.625  -7.375  -1.5     7.625]
 [-12.375  -3.5     2.625  13.5    26.75 ]
 [  3.25   14.875  22.625  20.125  14.625]
 [ 22.125  24.375  22.375  -0.125 -30.75 ]
 [ 13.625  12.625   9.75   -5.25  -26.25 ]]

Rounded Filtered Iy:
[[ -7  -7  -7  -2   8]
 [-12  -4   3  14  27]
 [  3  15  23  20  15]
 [ 22  24  22   0 -31]
 [ 14  13  10  -5 -26]]


## Step 3: $Ix^2$, $Iy^2$, and $IxIy$

Using element-wise multiplication

#### Calculation:
$I_x=
\begin{bmatrix}
-22 & -25  & -3 &   9 &   8 \\
-15 & -23  &  0 &  24 &  17 \\
 -5 & -12  &  3 &  29 &  19 \\
  0 &  -3  &  0 &   7 &   5 \\
 -1 &  -1  & -7 & -24 & -16
\end{bmatrix}
$

$I_x^2=
\begin{bmatrix}
(-22)^2 & (-25)^2  & (-3)^2 & (  9)^2 & (  8)^2 \\
(-15)^2 & (-23)^2  & ( 0)^2 & ( 24)^2 & ( 17)^2 \\
 (-5)^2 & (-12)^2  & ( 3)^2 & ( 29)^2 & ( 19)^2 \\
  (0)^2 & ( -3)^2  & ( 0)^2 & (  7)^2 & (  5)^2 \\
 (-1)^2 & ( -1)^2  & (-7)^2 & (-24)^2 & (-16)^2
\end{bmatrix}
$

In [6]:
Ix2, Iy2, Ixy = harris.compute_image_derivatives(I, image_width, image_height)

print("I_x^2:")
print(Ix2)

r_Ix2 = r_Ix**2
print("\nRounded I_x^2:")
print(r_Ix2)

I_x^2:
[[5.68139062e+03 2.64062500e+02 9.75156250e+01 2.25000000e+02
  1.17306250e+03]
 [1.01601562e+03 1.35056250e+03 2.75625000e+01 1.35976562e+03
  1.38756250e+03]
 [7.22500000e+01 4.10062500e+02 4.22500000e+01 3.49576562e+03
  1.35976562e+03]
 [2.13906250e+01 5.62500000e-01 6.40000000e+01 3.56265625e+02
  5.29000000e+02]
 [7.01406250e+01 0.00000000e+00 2.88906250e+01 2.58826562e+03
  2.29201562e+03]]

Rounded I_x^2:
[[484 625   9  81  64]
 [225 529   0 576 289]
 [ 25 144   9 841 361]
 [  0   9   0  49  25]
 [  1   1  49 576 256]]


In [7]:
print("I_y^2:")
print(Iy2)

r_Iy2 = r_Iy**2
print("\nRounded I_y^2:")
print(r_Iy2)

I_y^2:
[[3.45156250e+01 1.44000000e+02 1.96914062e+03 1.08900000e+03
  1.80625000e+01]
 [7.63140625e+02 2.25000000e+00 3.06250000e+00 5.58140625e+02
  1.70156250e+03]
 [4.90000000e+01 4.30562500e+02 1.54056250e+03 1.21626562e+03
  1.53076562e+03]
 [1.30501562e+03 6.50250000e+02 7.02250000e+02 1.18265625e+02
  4.79556250e+03]
 [4.56890625e+02 3.33062500e+02 8.92515625e+02 1.18265625e+02
  5.09439062e+03]]

Rounded I_y^2:
[[ 49  49  49   4  64]
 [144  16   9 196 729]
 [  9 225 529 400 225]
 [484 576 484   0 961]
 [196 169 100  25 676]]


In [8]:
print("$$I_xI_y$$:")
print(Ixy)
r_Ixy = r_Ix * r_Iy
print("\nRounded I_xy:")
print(r_Ixy)

$$I_xI_y$$:
[[  442.828125   195.        -438.203125   495.         145.5625  ]
 [  880.546875   -55.125        9.1875     871.171875  1536.5625  ]
 [  -59.5       -420.1875    -255.125     2061.984375  1442.734375]
 [  167.078125    19.125     -212.        -205.265625 -1592.75    ]
 [ -179.015625     0.         160.578125  -553.265625  3417.078125]]

Rounded I_xy:
[[ 154  175   21  -18   64]
 [ 180   92    0  336  459]
 [ -15 -180   69  580  285]
 [   0  -72    0    0 -155]
 [ -14  -13  -70  120  416]]


## Step 4: Gaussian on Image derivative

$Ixy=
\begin{bmatrix}
154 &  175  &  21  & -18 &   64 \\
180 &   92  &   0  & 336 &  459 \\
-15 & -180  &  69  & 580 &  285 \\
  0 &  -72  &   0  &   0 & -155 \\
-14 &  -13  & -70  & 120 &  416
\end{bmatrix}
$

#### Calculation

Look at the bottom left of $Ixy$, aka -168

$Ixy_{extended}=
\begin{bmatrix}
 ...& ... &  ... &  ... \\
\underline{ 0} & 0 &    -72 &  ... \\
\underline{-14} & \textbf{-14} &     13 &  ... \\
\underline{-14} & \underline{-14} &     \underline{13} &  ...
\end{bmatrix}
$

$
((-14)*(1+2+2+4) + 0 * 1 + 0 * 2 + (-13) * (1 + 2) + (-72)*1)/16 = -237 / 16 = -14.8125
$

In [9]:
g_r_Ix2 = harris.gaussian_filtering(r_Ix2, image_width, image_height)

print("Filtered g(I_x^2):")
print(g_r_Ix2)

r_g_r_Ix2 = round_matrix(g_r_Ix2)
print("\nRounded Filtered g(I_x^2):")
print(r_g_r_Ix2)

Filtered g(I_x^2):
[[464.6875 407.     204.8125 134.125  141.375 ]
 [294.     289.4375 246.0625 323.0625 317.6875]
 [103.1875 121.5625 198.0625 354.25   338.4375]
 [ 15.0625  25.625  112.125  234.6875 219.75  ]
 [  1.3125  10.875  130.1875 280.875  259.75  ]]

Rounded Filtered g(I_x^2):
[[465 407 205 134 141]
 [294 289 246 323 318]
 [103 122 198 354 338]
 [ 15  26 112 235 220]
 [  1  11 130 281 260]]


In [10]:
g_r_Iy2 = harris.gaussian_filtering(r_Iy2, image_width, image_height)

print("Filtered g(I_y^2):")
print(g_r_Iy2)

r_g_r_Iy2 = round_matrix(g_r_Iy2)
print("\nRounded Filtered g(I_y^2):")
print(r_g_r_Iy2)

Filtered g(I_y^2):
[[ 64.75    48.3125  42.6875  93.3125 185.6875]
 [ 84.      97.125  143.375  245.9375 377.3125]
 [186.25   267.5625 321.25   355.1875 463.5   ]
 [316.5625 366.375  322.8125 329.375  555.875 ]
 [268.6875 251.375  170.375  245.1875 565.125 ]]

Rounded Filtered g(I_y^2):
[[ 65  48  43  93 186]
 [ 84  97 143 246 377]
 [186 268 321 355 464]
 [317 366 323 329 556]
 [269 251 170 245 565]]


In [11]:
g_r_Ixy = harris.gaussian_filtering(r_Ixy, image_width, image_height)

print("Bottom left corner of I:", g_r_Ixy[4][0])

print("\nFiltered g(I_xy):")
print(g_r_Ixy)

r_g_r_Ixy = round_matrix(g_r_Ixy)
print("\nRounded Filtered g(I_xy):")
print(r_g_r_Ixy)

Bottom left corner of I: -14.8125

Filtered g(I_xy):
[[158.9375 121.1875  64.0625  79.875  139.6875]
 [104.75    59.1875  99.5625 239.0625 314.6875]
 [  6.875  -24.5     89.5    250.25   257.375 ]
 [-26.5    -44.      22.5625 111.875  117.0625]
 [-14.8125 -29.625  -10.6875 100.1875 227.4375]]

Rounded Filtered g(I_xy):
[[159 121  64  80 140]
 [105  59 100 239 315]
 [  7 -25  90 250 257]
 [-27 -44  23 112 117]
 [-15 -30 -11 100 227]]


## Step 5: Compute the Harris corner response image (C)

The processing window to 3x3 pixels and set the $\alpha$ value to 0.04.

#### Calculation

Look at the bottom left (element[4][0]) of $g(I_x^2)$, $g(I_y^2)$, $g(I_xy)$, aka 41, 594, -59 respectively

$$H=
\begin{bmatrix}
41 & -59 \\
-59 & 594
\end{bmatrix}
$$

Determinant: 
$
det(H) = (41 * 594) - (-59)(-59) = 20873 \\
$

Trace (sum of the main diagonal elements):
$
trace(H) = 41 + 594 = 635
$

Harris corner response:
$C = \text{det}(H) - \alpha(\text{trace}(H))^2 = 20873 - 0.04 * 635^2 = 4744$

In [12]:
alpha = 0.04
row, col = 4, 0

print("\nRounded I_x^2 at (4, 0):")
print(r_g_r_Ix2[row, col])
print("\nRounded I_y^2 at (4, 0):")
print(r_g_r_Iy2[row, col])
print("\nRounded I_xy at (4, 0):")
print(r_g_r_Ixy[row, col])
c_4_0 = harris.compute_single_cornerness_score(r_g_r_Ix2, r_g_r_Iy2, r_g_r_Ixy, row, col, alpha)

print(c_4_0, '\n')

row, col = 2, 2

print("\nRounded I_x^2 at (2, 2):")
print(r_g_r_Ix2[row, col])
print("\nRounded I_y^2 at (2, 2):")
print(r_g_r_Iy2[row, col])
print("\nRounded I_xy at (2, 2):")
print(r_g_r_Ixy[row, col])
c_2_2 = harris.compute_single_cornerness_score(r_g_r_Ix2, r_g_r_Iy2, r_g_r_Ixy, row, col, alpha)

print(c_2_2)


Rounded I_x^2 at (4, 0):
1

Rounded I_y^2 at (4, 0):
269

Rounded I_xy at (4, 0):
-15
-2872.0 


Rounded I_x^2 at (2, 2):
198

Rounded I_y^2 at (2, 2):
321

Rounded I_xy at (2, 2):
90
44683.56


In [13]:
c = harris.cornerness_score_matrix(r_g_r_Ix2, r_g_r_Iy2, r_g_r_Ixy, alpha, image_width, image_height)

print("Cornerness Score Matrix:")
print(c)

r_c = round_matrix(c)
print("\nRounded Cornerness Score Matrix:")
print(r_c)

Cornerness Score Matrix:
[[ 193338.64  466997.56  735091.36  686664.04  377875.36]
 [1741552.   1946692.76 2114301.44 1867762.   1574754.64]
 [1975662.76 2381777.   3259079.64 3848616.44 4372033.24]
 [ 616300.96 1366365.16 2972890.64 4534041.56 5758157.  ]
 [-119152.    698308.84 2307329.44 4235218.84 5755390.04]]

Rounded Cornerness Score Matrix:
[[ 193339  466998  735091  686664  377875]
 [1741552 1946693 2114301 1867762 1574755]
 [1975663 2381777 3259080 3848616 4372033]
 [ 616301 1366365 2972891 4534042 5758157]
 [-119152  698309 2307329 4235219 5755390]]


In [14]:
# The threshold value is 1500000. Set the corner response value below 1500000 to 0, and keep the value above 1500000 unchanged.
r_c_thresholded = np.where(r_c < 1500000, 0, r_c)

print("\nThresholded Rounded Cornerness Score Matrix:")
print(r_c_thresholded)


Thresholded Rounded Cornerness Score Matrix:
[[      0       0       0       0       0]
 [1741552 1946693 2114301 1867762 1574755]
 [1975663 2381777 3259080 3848616 4372033]
 [      0       0 2972891 4534042 5758157]
 [      0       0 2307329 4235219 5755390]]


## Non-Maximum Suppression

#### Calculation

Look at the value `5758157` using a 3x3 kernel, no value is greater than it, therefore, it remains, otherwise 0.
$$C=
\begin{bmatrix}
... & ... & ... \\
... & \underline{3848616} & \underline{4372033} \\
... & \underline{4534042} & \textbf{5758157} \\
... & \underline{4235219} & \underline{5755390}
\end{bmatrix}
$$

In [15]:
nms_c = harris.non_maximum_suppression(r_c_thresholded, image_width, image_height)

print("Non-Maximum Suppression Result:")
print(nms_c)

Non-Maximum Suppression Result:
[[      0.       0.       0.       0.       0.]
 [      0.       0.       0.       0.       0.]
 [      0.       0.       0.       0.       0.]
 [      0.       0.       0.       0. 5758157.]
 [      0.       0.       0.       0.       0.]]
